# Dev Pipeline

> Notebook generated from [`examples/subgraphs/03_dev_pipeline.py`](../../examples/subgraphs/03_dev_pipeline.py).

| Item | Value |
|------|-------|
| **Dataset** | GitHub Issues (embedded subset) |
| **API key** | ⚠️  **Requires API key** (`ANTHROPIC_API_KEY` or `OPENAI_API_KEY`) in `.env`. |

**Expected result:** PO → Architect → Developer → unit tests → QA → Reviewer → [4 gates]. Spec, generated code, tests, review score.

---

## Setup

```bash
uv pip install -e ".[dev,all]"
```

> To use `await main()` directly in cells, this notebook
> installs `nest_asyncio` in the first cell.

---

## Original docstring

```
Dev Pipeline Subgraph — Complete agile software development pipeline
=========================================================================
Subgraph: prismal.agents.subgraphs.dev_pipeline

Dataset: GitHub Issues + Feature Requests (real open-source projects)
  • Reference inspired by GitHub REST API issues:
    https://docs.github.com/en/rest/issues
  • Projects used as reference: FastAPI, Pydantic, LangChain, Typer
  • Why: GitHub Issues represent the natural input artifact
    for a development pipeline: they contain the requirement (PO), allow
    inferring architecture (Architect), generate code (Developer), unit tests
    (UnitTest), QA, and review (Reviewer) — exactly the 6
    agents of the Dev Pipeline.

Dev Pipeline subgraph description:
  po_agent → architect_agent → developer_agent → unit_test_agent
  → qa_agent → reviewer_agent → [quality gates]

  Gates (4):
  1. Test gate       : tests passing → continue; failing → retest (max 3)
  2. Review gate     : reviewer_score >= 0.8 → approval seed; < 0.8 → developer
  3. HITL seed       : writes artifact + risk_level into metadata
  4. HITL gate       : approve → END; reject/changes → developer (bypass if hitl=False)

  Parallelism:
  - Unit tests run in parallel by module (module_dispatcher)
  - dev_unit_tester_node runs in parallel for each module
  - dev_test_aggregator_node collects all results

Agents:
  po_agent        — Defines requirements, acceptance criteria, user stories
  architect_agent — Designs the architecture, interfaces, module structure
  developer_agent — Writes the implementation in code
  unit_test_agent — Writes unit tests for the generated code
  qa_agent        — QA, integration tests, regression
  reviewer_agent  — Code review, quality, score >= 0.8 to approve

Usage:
    uv run python examples/subgraphs/03_dev_pipeline.py
```

In [ ]:
# Enable nested event loop to use `await` directly in Jupyter.
import sys
if 'nest_asyncio' not in sys.modules:
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        %pip install -q nest_asyncio
        import nest_asyncio
        nest_asyncio.apply()

## Setup · imports and dataset

In [ ]:
from __future__ import annotations

import asyncio

from langchain_core.messages import HumanMessage

from prismal.agents.state import initial_state

# Import the subgraph with handling for optional dependencies
try:
    from prismal.agents.subgraphs.dev_pipeline.builder import (
        build_dev_pipeline_subgraph,
        register_dev_pipeline,
    )

    DEV_PIPELINE_AVAILABLE = True
except ImportError:
    DEV_PIPELINE_AVAILABLE = False

## Dataset: GitHub Issues from open-source projects

In [ ]:
# Real issues / inspired by FastAPI, Pydantic and LangChain.
GITHUB_ISSUES = [
    {
        "id": "GH-001",
        "repo": "prismal",
        "title": "Add rate limiting middleware for LLM provider calls",
        "labels": ["enhancement", "providers", "good-first-issue"],
        "priority": "high",
        "description": (
            "We need a rate limiting mechanism for LLM API calls to prevent "
            "hitting provider rate limits (tokens/min and requests/min). "
            "The solution should:\n"
            "- Support configurable limits per provider (Anthropic, OpenAI, etc.)\n"
            "- Implement token bucket or sliding window algorithm\n"
            "- Retry with exponential backoff on 429 errors\n"
            "- Expose metrics: current_rate, tokens_remaining, wait_time\n"
            "- Be async-native and thread-safe\n"
            "- Integrate with the existing ProviderRegistry\n\n"
            "Acceptance criteria:\n"
            "- Unit tests with >90% coverage\n"
            "- No breaking changes to existing ProviderRegistry API\n"
            "- Type-annotated with mypy strict compliance\n"
            "- Documentation in docstrings"
        ),
        "complexity": "medium",
        "estimated_hours": 8,
    },
    {
        "id": "GH-002",
        "repo": "prismal",
        "title": "Implement async context manager for AgentState checkpointing",
        "labels": ["enhancement", "core", "async"],
        "priority": "medium",
        "description": (
            "Currently, checkpointing requires manual calls to SqliteSaver. "
            "We should add an async context manager that automatically saves "
            "state at the beginning and end of each agent turn.\n\n"
            "Requirements:\n"
            "- `async with checkpoint_context(thread_id) as state:` pattern\n"
            "- Auto-save on __aexit__ (even on exception)\n"
            "- Support both SQLite and PostgreSQL backends\n"
            "- Compatible with LangGraph's existing checkpointer protocol\n"
            "- Performance: <5ms overhead per turn\n\n"
            "Acceptance criteria:\n"
            "- Integration tests with both backends\n"
            "- No memory leaks (context manager must release connections)\n"
            "- Works with Python 3.13 asyncio event loop"
        ),
        "complexity": "medium",
        "estimated_hours": 6,
    },
    {
        "id": "GH-003",
        "repo": "prismal",
        "title": "Create CLI tool for testing agent pipelines interactively",
        "labels": ["enhancement", "cli", "dx"],
        "priority": "low",
        "description": (
            "Developers need a way to test agent pipelines from the terminal "
            "without writing Python scripts. Implement a `prismal run` CLI "
            "command using Typer.\n\n"
            "Features:\n"
            "- `prismal run <pipeline> --input 'user message'`\n"
            "- `prismal run --stream` for token-by-token output\n"
            "- `prismal list-pipelines` to enumerate registered subgraphs\n"
            "- JSON output mode: `--format json`\n"
            "- Thread ID support: `--thread-id my-session`\n\n"
            "Tech stack: Typer + Rich for beautiful output\n\n"
            "Acceptance criteria:\n"
            "- Works on Linux, macOS, Windows\n"
            "- Tests using Click test runner\n"
            "- `--help` for all commands"
        ),
        "complexity": "low",
        "estimated_hours": 4,
    },
]

## Pipeline configuration

In [ ]:
def format_issue_as_task(issue: dict) -> str:
    """Convert a GitHub Issue into an instruction for the Dev Pipeline."""
    return (
        f"Issue #{issue['id']}: {issue['title']}\n"
        f"Repository: {issue['repo']}\n"
        f"Priority: {issue['priority']} | Complexity: {issue['complexity']}\n"
        f"Labels: {', '.join(issue['labels'])}\n\n"
        f"Full description:\n{issue['description']}\n\n"
        f"Estimated time: {issue['estimated_hours']} hours\n\n"
        "Develop this feature following the full pipeline:\n"
        "1. PO: define user stories and acceptance criteria\n"
        "2. Architect: design architecture and interfaces\n"
        "3. Developer: implement the code in Python\n"
        "4. UnitTest: write unit tests with pytest\n"
        "5. QA: verify coverage and edge cases\n"
        "6. Reviewer: review quality, security and style"
    )


async def run_dev_pipeline_real(issue: dict) -> dict:
    """Run the real Dev Pipeline with the compiled subgraph."""
    await register_dev_pipeline()
    subgraph_def = build_dev_pipeline_subgraph()

    from prismal.agents.subgraphs.factory import SubgraphFactory

    compiled = SubgraphFactory.compile(subgraph_def)

    state = initial_state()
    state["messages"] = [HumanMessage(content=format_issue_as_task(issue))]
    state["metadata"] = {
        "issue_id": issue["id"],
        "priority": issue["priority"],
        "hitl_enabled": False,  # disable human approval in the example
        "dev_pipeline": {
            "iteration": 0,
            "review_attempts": 0,
        },
    }

    config = {
        "configurable": {
            "thread_id": f"dev_{issue['id']}",
            "recursion_limit": 25,
        }
    }

    return await compiled.ainvoke(state, config=config)


def simulate_dev_pipeline(issue: dict) -> None:
    """Simulate execution of the Dev Pipeline by showing the expected output."""
    print("\n  [Node 1: PO Agent]")
    print(f"    Analyzing issue #{issue['id']}...")
    print("    → User Stories defined:")
    print(f"      AS a developer, I WANT {issue['title'].lower()}")
    print("      SO THAT the system handles rate limits gracefully")
    print(
        f"    → Acceptance criteria extracted: {len(issue['description'].split(chr(10)))} lines"
    )

    print("\n  [Node 2: Architect Agent]")
    print(f"    Designing architecture for: {issue['title'][:50]}...")
    arch_patterns = {
        "rate limiting": "Token Bucket + CircuitBreaker pattern",
        "context manager": "AsyncContextManager + Repository pattern",
        "CLI tool": "Command pattern + Observer (streaming)",
    }
    pattern = next(
        (v for k, v in arch_patterns.items() if k in issue["title"].lower()),
        "Adapter + Factory pattern",
    )
    print(f"    → Architectural pattern: {pattern}")
    print("    → Modules identified: 3-4 modules")
    print("    → Interfaces defined: Protocol classes with type hints")

    print("\n  [Node 3: Developer Agent]")
    print("    Implementing code...")
    complexity_map = {"high": "~250 lines", "medium": "~150 lines", "low": "~80 lines"}
    lines = complexity_map.get(issue["complexity"], "~120 lines")
    print(f"    → Code generated: {lines} of Python")
    print("    → Type annotations: 100% (mypy strict)")
    print("    → Docstrings: Google style on all public functions")

    print("\n  [Node 4: Unit Test Agent]")
    cov = {"high": 94, "medium": 91, "low": 88}[issue["complexity"]]
    print("    Writing tests with pytest + pytest-asyncio...")
    print(f"    → Tests generated: {cov // 10 + 2} test cases")
    print(f"    → Estimated coverage: {cov}%")
    print("    → LLM provider mock: yes (no real calls)")

    print("\n  [Node 4b: Parallel Unit Tester] (modules in parallel)")
    n_modules = {"high": 4, "medium": 3, "low": 2}[issue["complexity"]]
    print(f"    module_dispatcher → {n_modules} parallel workers")
    for i in range(1, n_modules + 1):
        print(f"      Worker {i}: tests for module_{i} → PASS ✓")
    print(f"    dev_test_aggregator → {n_modules}/{n_modules} modules OK")

    print("\n  [Test Gate]")
    print("    → All tests pass ✓ → proceed to QA")

    print("\n  [Node 5: QA Agent]")
    print("    Integration tests and edge cases...")
    print("    → Integration tests: PASS ✓")
    print("    → Edge cases tested: empty inputs, timeouts, network errors")
    print("    → Regression with full suite: PASS ✓")

    print("\n  [Node 6: Reviewer Agent]")
    review_score = {"high": 0.88, "medium": 0.85, "low": 0.92}[issue["complexity"]]
    print("    Code review with quality checklist...")
    print(f"    → Quality score: {review_score:.2f}/1.0")
    passed = review_score >= 0.8
    print(f"    → Review gate (>= 0.8): {'PASS ✓' if passed else 'FAIL ✗ → back to Developer'}")

    print("\n  [Review Gate → HITL Approval Seed]")
    print("    Artifact saved at metadata['dev_pipeline']['code_artifact']")
    print(f"    Risk level: {'HIGH' if issue['priority'] == 'high' else 'MEDIUM'}")

    print("\n  [HITL Gate] (hitl_enabled=False → automatic bypass)")
    print("    → Automatic approval → END ✓")

    print(f"\n  Pipeline completed for issue #{issue['id']}")


async def main() -> None:
    print("=" * 70)
    print("  Dev Pipeline Subgraph — Dataset: GitHub Issues (prismal)")
    print("=" * 70)

    # Show subgraph architecture
    print("\n[Dev Pipeline architecture]")
    nodes = [
        ("po_agent        ", "Defines requirements, user stories, acceptance criteria"),
        ("architect_agent ", "Designs architecture, interfaces, module structure"),
        ("developer_agent ", "Writes Python implementation (can iterate max 3×)"),
        ("unit_test_agent ", "Writes pytest tests with target coverage >90%"),
        ("[Parallel Tests] ", "module_dispatcher → N workers → aggregator"),
        ("[Test Gate]      ", "All tests pass? YES → QA | NO → unit_test (max 3×)"),
        ("qa_agent        ", "Integration tests, edge cases, regression"),
        ("reviewer_agent  ", "Code review: score >= 0.8 → approval | < 0.8 → developer"),
        ("[Review Gate]   ", "score >= 0.8? YES → HITL seed | NO → developer"),
        ("[HITL Seed]     ", "Writes artifact + risk_level into metadata"),
        ("[HITL Gate]     ", "approve → END | reject → developer | bypass if hitl=False"),
    ]
    for node, desc in nodes:
        print(f"  {node}: {desc}")

    print(f"\n[Issues to develop: {len(GITHUB_ISSUES)}]")
    for issue in GITHUB_ISSUES:
        print(f"  #{issue['id']}: {issue['title'][:55]}... [{issue['priority'].upper()}]")

    # Run the pipeline for each issue
    print("\n" + "─" * 70)
    for issue in GITHUB_ISSUES:
        print(f"\n{'═' * 70}")
        print(f"  Feature: {issue['title']}")
        print(
            f"  Issue: #{issue['id']} | Priority: {issue['priority']} | Complexity: {issue['complexity']}"
        )
        print(f"  Estimate: {issue['estimated_hours']}h")
        print(f"{'─' * 70}")

        if DEV_PIPELINE_AVAILABLE:
            try:
                final_state = await run_dev_pipeline_real(issue)
                messages = final_state.get("messages", [])
                pipeline_meta = final_state.get("metadata", {}).get("dev_pipeline", {})

                print("\n  [Real pipeline result]")
                if pipeline_meta:
                    print(f"    Development iterations : {pipeline_meta.get('iteration', 1)}")
                    print(
                        f"    Review score           : {pipeline_meta.get('review_result', {}).get('score', 'N/A')}"
                    )
                    print(
                        f"    Artifact generated     : {bool(pipeline_meta.get('code_artifact'))}"
                    )

                if messages:
                    last_msg = str(messages[-1].content)
                    print("\n  Final output:")
                    print(f"  {last_msg[:400]}")

            except Exception as exc:
                print(f"  [Simulated mode — reason: {type(exc).__name__}]")
                simulate_dev_pipeline(issue)
        else:
            print("  [Simulated mode — subgraph not available]")
            simulate_dev_pipeline(issue)

    # Summary and comparison with manual development
    print(f"\n{'═' * 70}")
    print("  COMPARISON: Dev Pipeline vs Manual Development")
    print(f"{'═' * 70}")

    total_hours = sum(i["estimated_hours"] for i in GITHUB_ISSUES)
    print(f"\n  {'Phase':<25} {'Manual':>10} {'Dev Pipeline':>14}")
    print("  " + "─" * 52)
    phases = [
        ("Requirements definition ", "2h/feature", "automatic (PO Agent)"),
        ("Architecture design     ", "3h/feature", "automatic (Arch Agent)"),
        ("Implementation          ", f"{total_hours}h total", "parallel + iterative"),
        ("Unit tests              ", "2h/feature", "automatic (UT Agent)"),
        ("QA / Integration        ", "1h/feature", "automatic (QA Agent)"),
        ("Code review             ", "1h/feature", "automatic score >= 0.8"),
    ]
    for phase, manual, pipeline in phases:
        print(f"  {phase}: {manual:>10} → {pipeline}")

    print("\n  Quality Gates guarantee:")
    print("    ✓ Tests always pass before QA")
    print("    ✓ Code review score >= 0.8 before approval")
    print("    ✓ Optional human approval (HITL) for critical changes")
    print("    ✓ Maximum 3 iterations per gate (anti-infinite-loop)")

    print("\n[HITL Configuration]")
    print("  hitl_enabled=True  → pause and wait for human approval")
    print("  hitl_enabled=False → automatic bypass (CI/CD mode)")
    print("  risk_level HIGH/MEDIUM/LOW → visible in the interrupt payload")


if __name__ == "__main__":
    asyncio.run(main())

---

## Run the demo

The original file ends with `asyncio.run(main())`. In Jupyter use:

```python
await main()
```

Thanks to the `nest_asyncio.apply()` in the first cell, the
`async` calls run without needing a separate event loop.

In [ ]:
# Uncomment to run the end-to-end demo (requires API key if applicable).
# await main()